# 01 · vla_ros2 quickstart (CPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/vla_ros2/blob/main/notebooks/01_quickstart.ipynb)

Install the Python runtime and run the always-available adapters (`dummy`, `random`,
`scripted`). No GPU, no model weights, no ROS2. This is the same adapter boundary the
ROS2 node (`vla_runtime_node`) calls on the robot.

In [ ]:
# Colab: clone the repo. Locally: run from the repo (or its notebooks/ folder).
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("vla_ros2").exists():
    !git clone --depth 1 https://github.com/rsasaki0109/vla_ros2.git
REPO = Path("vla_ros2").resolve() if IN_COLAB else Path.cwd()
if not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
%pip install -q -e .

In [ ]:
!vla-ros2 list

## One prediction through the typed contract

In [ ]:
import numpy as np

from vla_ros2 import load_model
from vla_ros2.core.types import VLAObservation

observation = VLAObservation(
    instruction="stack the red block on the blue block",
    images={"primary": np.zeros((256, 256, 3), dtype=np.uint8)},
    state={"state": [0.0, 45.0, 90.0, 0.0, 0.0, 0.5]},
)
model = load_model("scripted")
action = model.predict_observation(observation)
print(action.spec.action_space, action.spec.shape)
print(np.round(action.data, 3))

## Same input through several adapters

In [ ]:
from vla_ros2.compare import compare_models

for result in compare_models(["dummy", "random", "scripted"], observation):
    row = result.to_row()
    print(f"{row['model']:>9}  ok={row['ok']}  infer={row['infer_ms']} ms  {row['action_preview']}")

Next: [02 · SmolVLA inference on GPU](02_smolvla_inference.ipynb) ·
[03 · offline evaluation](03_offline_eval.ipynb) ·
[04 · VLM success detector](04_vlm_success.ipynb)